# NeMo Gym + Fireworks `async_rl_loop`: Setup and Run

Run this notebook top to bottom -- **Run All** -- with nothing pre-cloned or pre-installed, to
reproduce a **confirmed-working, live-verified** GRPO reinforcement-fine-tuning loop where
**NeMo Gym controls the RL environment** (task generation, agent harness, verification) and
**Fireworks AI** runs training and inference, wired together through Fireworks' own
`training.recipes.async_rl_loop.main()` framework.

This is the single entrypoint for this repo. Switching **environment** or **model** is a config
swap (`ENV_CONFIG` / `MODEL_CONFIG` below), not a new notebook. For the engineering narrative --
why this is built the way it is, what's held constant, what's still open -- see
[`DESIGN.md`](DESIGN.md). For the evidence -- every confirmed run, exact numbers, logs -- see
[`RESULTS.md`](RESULTS.md).

**What you actually need before starting:**
- A **Fireworks API key** (`FIREWORKS_API_KEY`) from a billing-enabled account -- prompted for
  via `getpass` below, never hardcoded.
- **`git`** on `PATH` (to clone the two upstream repos). This is the one thing this notebook
  cannot install for you.
- `uv` is auto-installed if missing (§1).
- **One manual, one-time step (§3)**: apply the two patch files shipped next to this notebook
  via `git apply`. See `DESIGN.md` for what each one does and why it's a patch rather than an
  upstream change. Everything else -- both venvs, the NeMo Gym environment, the training run
  itself -- is fully automated in the cells below.

**This provisions a real Fireworks Dedicated trainer + inference deployment once you flip the
safety switch below -- roughly $40-100/hr combined while both are up, depending on model size
(see `RESULTS.md` §Cost).** `cleanup_on_exit=True` plus the async loop's own circuit breaker
reliably tear both down within minutes of any failure or on natural completion. On macOS the
training cell runs under `caffeinate -i` so a sleeping machine can't turn a short run into a
long, expensive one.


## 1. Prerequisites

In [ ]:
import shutil
import subprocess
import sys
import platform
import os
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

assert shutil.which("git"), "git is required and not on PATH -- install it before continuing."
print("git:", shutil.which("git"))

# NeMo Gym's own CLI builds shell commands like `cd {dir_path} && ...` without
# quoting the path (nemo_gym/cli/setup_command.py, nemo_gym/cli/env.py) -- a
# working directory with spaces breaks every server it launches with a
# confusing "cd: too many arguments" / "Process ... finished unexpectedly"
# failure. Not something this notebook can patch around; fail fast here
# instead of downstream in a Ray traceback.
assert " " not in str(Path.cwd()), (
    f"Run this notebook from a path with no spaces -- got {Path.cwd()!r}. "
    "NeMo Gym's own CLI does not quote paths in the shell commands it spawns "
    "servers with, and a space in the path breaks every server it launches."
)

if not shutil.which("uv"):
    print("uv not found -- installing via the official installer...")
    subprocess.run(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True, check=True,
    )
    # The installer places uv in ~/.local/bin, which may not be on this process's PATH yet.
    local_bin = str(Path.home() / ".local" / "bin")
    if local_bin not in os.environ.get("PATH", "") and Path(local_bin).exists():
        os.environ["PATH"] = local_bin + os.pathsep + os.environ.get("PATH", "")

uv_path = shutil.which("uv")
assert uv_path, "uv still not found after install -- add ~/.local/bin to PATH and re-run this cell."
print("uv:", uv_path)
out = subprocess.run(["uv", "--version"], capture_output=True, text=True)
print(" ", out.stdout.strip())


## 2. Clone the two upstream repos

In [ ]:
HERE = Path.cwd()
NEMO_GYM_DIR = HERE / "nemo-gym"
COOKBOOK_DIR = HERE / "cookbook"

if NEMO_GYM_DIR.exists():
    print(f"Already present: {NEMO_GYM_DIR}")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/NVIDIA-NeMo/Gym.git", str(NEMO_GYM_DIR)],
        check=True,
    )

if COOKBOOK_DIR.exists():
    print(f"Already present: {COOKBOOK_DIR}")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/fw-ai/cookbook.git", str(COOKBOOK_DIR)],
        check=True,
    )

print(sorted(p.name for p in NEMO_GYM_DIR.iterdir())[:10])
print(sorted(p.name for p in COOKBOOK_DIR.iterdir())[:10])


## 3. Apply this repo's patches -- required, one-time, manual

**You need to do this yourself before continuing.** The bridge code
(`proxy.py`, `train.py`) and a small `inference_provider` fix are not part of either upstream
repo -- a fresh clone (§2) does not include them. See [`DESIGN.md`](DESIGN.md) for exactly what
each patch does and why. Run, from a terminal, from wherever this notebook lives:

```bash
cd cookbook && git apply ../nemo_gym_multistep_new_files.patch && cd ..
cd nemo-gym && git apply ../inference_provider_rollout_id.patch && cd ..
```

The verification cell below confirms both patches were applied before you continue.


In [ ]:
nemo_gym_multistep_dir = COOKBOOK_DIR / "training" / "examples" / "rl" / "nemo_gym_multistep"
app_py_path = NEMO_GYM_DIR / "responses_api_models" / "inference_provider" / "app.py"

missing = []
if not (nemo_gym_multistep_dir / "proxy.py").exists():
    missing.append("cookbook: nemo_gym_multistep_new_files.patch not applied (proxy.py missing)")
if not (nemo_gym_multistep_dir / "train.py").exists():
    missing.append("cookbook: nemo_gym_multistep_new_files.patch not applied (train.py missing)")
if "_CaptureRolloutIdMiddleware" not in app_py_path.read_text():
    missing.append("nemo-gym: inference_provider_rollout_id.patch not applied (app.py unpatched)")

if missing:
    raise SystemExit(
        "Required patches not applied yet -- see the instructions above.\n  " + "\n  ".join(missing)
    )
print("Both patches applied correctly.")


## 4. Build both Python environments

In [ ]:
def run_streamed(cmd, cwd):
    print(f"$ {' '.join(cmd)}  (cwd={cwd})")
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{cmd[0]} failed in {cwd} (exit {result.returncode})")

# Best-effort -- some uv installs (e.g. via a system package manager) don't support self
# update; that's fine, sync still works against whatever version is present.
subprocess.run(["uv", "self", "update"], cwd=NEMO_GYM_DIR, capture_output=True)

run_streamed(["uv", "sync"], cwd=NEMO_GYM_DIR)
run_streamed(["uv", "sync"], cwd=COOKBOOK_DIR / "training")

assert (NEMO_GYM_DIR / ".venv" / "bin" / "gym").exists(), "nemo-gym/.venv missing the `gym` CLI after uv sync"
assert (COOKBOOK_DIR / "training" / ".venv" / "bin" / "python3").exists(), "cookbook/training/.venv missing after uv sync"
print("Both venvs ready.")


## 5. Fireworks credentials

In [ ]:
import getpass
import json as _json
import urllib.request

if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("FIREWORKS_API_KEY: ")

def _fw_get(path):
    req = urllib.request.Request(
        f"https://api.fireworks.ai/v1{path}",
        headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"},
    )
    with urllib.request.urlopen(req) as resp:
        return _json.load(resp)

accounts = _fw_get("/accounts")["accounts"]
assert accounts, "No accounts visible for this API key -- check it's valid and billing-enabled."
FIREWORKS_ACCOUNT_ID = accounts[0]["name"].rsplit("/", 1)[-1]
os.environ["FIREWORKS_ACCOUNT_ID"] = FIREWORKS_ACCOUNT_ID
print("Resolved account:", FIREWORKS_ACCOUNT_ID)


## 6. Architecture

See [`DESIGN.md`](DESIGN.md) for the full narrative -- ownership split, the two integration
points, and the key design decisions. Short version:

```
                        NeMo Gym owns this box                     Fireworks owns this box
        +-------------------------------------------------+   +--------------------------------+
        |  resources server   agent harness   model server |   |  Dedicated trainer (GRPO)      |
        |  (verifier, tools)  (e.g. simple_agent) (inference_ |   |  Dedicated inference deployment |
        |                                          provider,   |<--+  (hotloaded every step)         |
        |                                          patched)    |   |                                  |
        +-------------------------------------------------+   +--------------------------------+
                        |                          |
                        |  /run (direct HTTP)       |  proxy.py: TinkerRecordingProxy
                        |  from rollout_fn           |  (OpenAI-compatible HTTP server,
                        v                          v   samples from the live deployment,
              async_rl_loop.main()  <----------------  records exact tokens/logprobs)
              (GRPO groups, optimizer,
               checkpointing, hotload)
```

`train.py`'s `rollout_fn` makes one direct HTTP `/run` POST per sample straight to the live
NeMo Gym agent harness, then drains the matching `proxy.py` session into a `RolloutRun`/
`RolloutSample` for the recipe. Which environment and harness that `/run` call targets is set
by §7 below.


## 7. Choose an environment

This is what makes this repo a one-stop shop rather than a single fixed demo: switching
environments is this block, not a new notebook or any code change in `proxy.py`/`train.py`. See
[`RESULTS.md`](RESULTS.md) for every environment this has actually been run against, including
known caveats per environment.


In [ ]:
# --- example_multi_step: short, simple multi-turn tool-calling task (confirmed working) ---
ENV_CONFIG = {
    "resources_server": "example_multi_step",
    "agent_name": "example_multi_step_simple_agent",
    "dataset_path": None,  # None -> train.py resolves the env's own bundled example.jsonl
}

# --- workplace_assistant: longer agentic CRM+calendar task, 27 tools (confirmed working) ---
# ENV_CONFIG = {
#     "resources_server": "workplace_assistant",
#     "agent_name": "workplace_assistant_simple_agent",
#     "dataset_path": None,
# }

# --- toolsandbox: different agent harness (toolsandbox_agent, not simple_agent), dual-role
#     model serving (policy model also plays the simulated user), genuinely continuous
#     [0,1] reward (confirmed working -- see RESULTS.md for the dual-role routing story) ---
# ENV_CONFIG = {
#     "resources_server": "toolsandbox",
#     "agent_name": "toolsandbox_agent",
#     "dataset_path": None,
# }

print("Using resources server:", ENV_CONFIG["resources_server"])


In [ ]:
RUN_DIR = COOKBOOK_DIR / "training" / "examples" / "rl" / "nemo_gym_multistep"
dataset_path = ENV_CONFIG["dataset_path"] or (
    NEMO_GYM_DIR / "resources_servers" / ENV_CONFIG["resources_server"] / "data" / "example.jsonl"
)
rows = [_json.loads(line) for line in dataset_path.read_text().splitlines() if line.strip()]
print(f"{len(rows)} rows in {dataset_path}")

row = rows[0]
user_messages = [m for m in row["responses_create_params"]["input"] if m.get("role") == "user"]
if user_messages:
    print("Example user query:", user_messages[0]["content"])
else:
    # toolsandbox's rows are just {"task_idx": N} -- the actual scenario (system
    # prompt, user turns, tool schemas) is generated server-side from task_idx,
    # not stored in the dataset row itself.
    print("Example row (scenario generated server-side from task_idx):", row)


## 8. Free sanity check: does the environment boot?

Before spending anything live, confirm the chosen environment's 3 servers actually come up.
This costs nothing -- `inference_provider` just needs `policy_base_url` to be set for its config
to resolve; nothing needs to actually be listening on it yet. `train.py` writes the real one
(pointing at `TinkerRecordingProxy`) before its own `gym env start` call -- this is the same
`env.yaml` shape with a placeholder URL, just to prove the environment itself boots.


In [ ]:
import time

capture_dir = NEMO_GYM_DIR / "results" / "model_call_capture"
(NEMO_GYM_DIR / "env.yaml").write_text(
    "# env.yaml -- placeholder written by this notebook's free readiness demo\n"
    "policy_base_url: http://127.0.0.1:18234/v1\n"
    'policy_api_key: "unused"\n'
    "policy_model_name: policy\n"
    "observability_enabled: true\n"
    f"model_call_capture_dir: {capture_dir}\n"
)

demo_log = RUN_DIR / "gym_env_demo.log"
with open(demo_log, "w") as f:
    demo_proc = subprocess.Popen(
        [str(NEMO_GYM_DIR / ".venv" / "bin" / "gym"), "env", "start",
         "--resources-server", ENV_CONFIG["resources_server"], "--model-type", "inference_provider", "-v"],
        cwd=NEMO_GYM_DIR, stdout=f, stderr=subprocess.STDOUT,
    )

print(f"gym env start launched (pid={demo_proc.pid}), waiting for readiness...")
deadline = time.time() + 120
ready = False
while time.time() < deadline:
    text = demo_log.read_text(errors="replace")
    if "All 3 / 3 servers ready" in text:
        ready = True
        break
    if demo_proc.poll() is not None:
        break
    time.sleep(2)

print("All 3 servers ready:", ready)
if not ready:
    print(f"--- see {demo_log} for what happened ---")

demo_proc.terminate()
try:
    demo_proc.wait(timeout=30)
except subprocess.TimeoutExpired:
    demo_proc.kill()
print("Demo environment stopped.")


## 9. Choose a model

In [ ]:
# --- Dense: Qwen 3.5 27B (confirmed working) ---
MODEL_CONFIG = {
    "base_model": "accounts/fireworks/models/qwen3p5-27b",
    "tokenizer_model": "Qwen/Qwen3.5-27B",
    "training_shape_id": "accounts/fireworks/trainingShapes/qwen3p5-27b-64k-lora",
}

# --- MoE: Qwen 3.5 35B-A3B, 35B total / 3B active params (confirmed working) ---
# MODEL_CONFIG = {
#     "base_model": "accounts/fireworks/models/qwen3p5-35b-a3b",
#     "tokenizer_model": "Qwen/Qwen3.5-35B-A3B",
#     "training_shape_id": "accounts/fireworks/trainingShapes/qwen3p5-35b-a3b-256k-lora",
# }

print("Using:", MODEL_CONFIG["base_model"])


## 10. Run the training loop -- real cost starts here

Flip `RUN_IT = True` below to actually run. This provisions a real Fireworks Dedicated trainer
+ inference deployment for `ENV_CONFIG` x `MODEL_CONFIG` chosen above. See `RESULTS.md` for
what to expect and roughly what it costs.


In [ ]:
RUN_IT = False

def guarded(what):
    if not RUN_IT:
        print(f"Skipping {what} -- set RUN_IT = True above to actually run it (this costs real money).")
        return False
    return True

if guarded("async_rl_loop training run"):
    prefix = ["caffeinate", "-i"] if sys.platform == "darwin" else []
    cmd = prefix + [
        str(COOKBOOK_DIR / "training" / ".venv" / "bin" / "python3"),
        "-m", "training.examples.rl.nemo_gym_multistep.train",
        "--resources-server", ENV_CONFIG["resources_server"],
        "--agent-name", ENV_CONFIG["agent_name"],
        "--base-model", MODEL_CONFIG["base_model"],
        "--tokenizer-model", MODEL_CONFIG["tokenizer_model"],
        "--training-shape-id", MODEL_CONFIG["training_shape_id"],
        "--max-rows", "5",
        "--completions-per-prompt", "4",
        "--prompt-groups-per-step", "2",
        "--max-completion-tokens", "256",
        "--run-timeout-s", "180",
    ]
    if ENV_CONFIG["dataset_path"]:
        cmd += ["--dataset-path", str(ENV_CONFIG["dataset_path"])]
    # train.py logs via logging.basicConfig(), which defaults to stderr, not
    # stdout -- every INFO line (trainer state, rollout rewards, the trainer
    # job id, "Async RL training complete", the checkpoint name) lives there.
    # Merge streams so nothing gets silently dropped on either a successful
    # or a failed run.
    result = subprocess.run(cmd, cwd=COOKBOOK_DIR, env=os.environ, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-12000:])
    print(f"\n[exit code: {result.returncode}]")


## 11. Scaling up beyond the smoke test

The config above is intentionally minimal (5 rows, 3 optimizer steps, 256 max completion
tokens) -- enough to prove the pipeline is mechanically correct, not to produce a meaningfully
improved model. To scale up:

1. **More data.** Point `--dataset-path` at a larger JSONL (or leave `ENV_CONFIG["dataset_path"]`
   as `None` and increase `--max-rows`). `workplace_assistant`'s own config already references a
   1260-prompt HuggingFace dataset (`nvidia/Nemotron-RL-agent-workplace_assistant`) -- see
   `DESIGN.md`'s open items.
2. **More steps.** Raise `--prompt-groups-per-step` and let the loop run longer; watch cost.
3. **More completions per prompt.** `--completions-per-prompt` controls GRPO group size --
   higher gives a less noisy advantage estimate at proportionally higher cost.
4. **Longer completions.** `--max-completion-tokens` caps response length; raise it for tasks
   that need longer reasoning or tool-call chains.
5. **A before/after eval.** Nothing here currently evaluates the trained checkpoint against a
   held-out set -- add an eval pass against `--output-model-id` after training to know whether
   the run actually helped.
6. **A different training shape** for a bigger base model -- see `MODEL_CONFIG` in §9 and
   `RESULTS.md` for provisioning-time notes on larger shapes.

Also pass `--output-model-id <name>` to promote the final checkpoint to a durable named model --
without it, `cleanup_on_exit=True` deletes the trainer job once training completes, and an
un-captured job id makes the checkpoint effectively unreachable (checkpoints survive job
deletion, but only if you have the job id, which is only visible in this cell's log output).


## 12. Teardown verification

In [ ]:
deployments = _fw_get(f"/accounts/{FIREWORKS_ACCOUNT_ID}/deployments")
print("Active deployments:", deployments.get("totalSize", 0))
for dep in deployments.get("deployments", []):
    print(" -", dep.get("name"), dep.get("state"))
assert deployments.get("totalSize", 0) == 0, \
    "A deployment is still running -- delete it manually via the Fireworks console or API."
print("Clean -- nothing left running.")


## 12b. Optional capstone: promote, deploy on-demand, compare base vs. fine-tuned

Everything above uses `async_rl_loop.main()`'s own SDK-managed rollout deployment. This
section exercises a **second, independent Fireworks API** -- the
[Deployments API](https://docs.fireworks.ai/guides/ondemand-deployments) -- against the
checkpoint the training run above just produced (requires `--output-model-id` to have been
set so the checkpoint was promoted to a durable model; see `RESULTS.md` for what to expect).

This costs a small additional amount on top of the training run (~3-4 minutes of on-demand
GPU time per deployment, one deployment at a time, torn down immediately after querying) --
see `RESULTS.md`'s "Base vs. fine-tuned" section for what this actually demonstrated at
smoke-test scale (a pipeline check, not a fine-tuning quality eval).


In [ ]:
RUN_CAPSTONE = False

OUTPUT_MODEL_ID = None  # set this to the --output-model-id you trained with above

def guarded_capstone(what):
    if not RUN_CAPSTONE:
        print(f"Skipping {what} -- set RUN_CAPSTONE = True (and OUTPUT_MODEL_ID) above to run it.")
        return False
    if not OUTPUT_MODEL_ID:
        print("Set OUTPUT_MODEL_ID to the model id you trained with --output-model-id.")
        return False
    return True

if guarded_capstone("base-vs-fine-tuned comparison"):
    import time as _time

    def _deploy_on_demand(base_model, deployment_id, accelerator_count=4):
        # Inference deployments require a power-of-2 world size -- the training
        # shape's accelerator count (e.g. 3x B200) is NOT necessarily valid here.
        body = {
            "baseModel": base_model,
            "acceleratorType": "NVIDIA_B200_180GB",
            "acceleratorCount": accelerator_count,
            "minReplicaCount": 1,
            "maxReplicaCount": 1,
        }
        req = urllib.request.Request(
            f"https://api.fireworks.ai/v1/accounts/{FIREWORKS_ACCOUNT_ID}/deployments?deploymentId={deployment_id}",
            data=_json.dumps(body).encode(),
            headers={
                "Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req) as resp:
            return _json.load(resp)

    def _wait_ready(deployment_id, timeout_s=480):
        deadline = time.time() + timeout_s
        while time.time() < deadline:
            state = _fw_get(f"/accounts/{FIREWORKS_ACCOUNT_ID}/deployments/{deployment_id}").get("state")
            if state in ("READY", "FAILED"):
                return state
            _time.sleep(10)
        raise TimeoutError(f"{deployment_id} did not reach a terminal state in {timeout_s}s")

    def _query(model_ref, messages, tools):
        body = {"model": model_ref, "messages": messages, "tools": tools, "max_tokens": 512, "temperature": 0.0}
        req = urllib.request.Request(
            "https://api.fireworks.ai/inference/v1/chat/completions",
            data=_json.dumps(body).encode(),
            headers={
                "Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req) as resp:
            return _json.load(resp)

    def _delete_deployment(deployment_id):
        req = urllib.request.Request(
            f"https://api.fireworks.ai/v1/accounts/{FIREWORKS_ACCOUNT_ID}/deployments/{deployment_id}?ignore_checks=true",
            headers={"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}"},
            method="DELETE",
        )
        with urllib.request.urlopen(req) as resp:
            resp.read()

    # A held-out prompt -- reuses the first ENV_CONFIG row's system+user text and tool schemas.
    prompt_row = rows[0]
    input_items = prompt_row["responses_create_params"]["input"]
    messages = [{"role": m["role"], "content": m["content"]} for m in input_items if m.get("role") in ("system", "user")]
    tools = [
        {"type": "function", "function": {"name": t["name"], "description": t.get("description", ""), "parameters": t["parameters"]}}
        for t in prompt_row["responses_create_params"].get("tools", [])
    ]

    base_model_id = MODEL_CONFIG["base_model"]
    finetuned_model_id = f"accounts/{FIREWORKS_ACCOUNT_ID}/models/{OUTPUT_MODEL_ID}"

    print("Deploying base model on-demand...")
    _deploy_on_demand(base_model_id, "capstone-base-cmp")
    state = _wait_ready("capstone-base-cmp")
    print("base deployment state:", state)
    if state == "READY":
        base_result = _query(f"{base_model_id}#accounts/{FIREWORKS_ACCOUNT_ID}/deployments/capstone-base-cmp", messages, tools)
        print("base response:", _json.dumps(base_result.get("choices", [{}])[0].get("message", {}), indent=2))
    _delete_deployment("capstone-base-cmp")

    print("\nDeploying fine-tuned model on-demand (live-merge)...")
    _deploy_on_demand(finetuned_model_id, "capstone-finetuned-cmp")
    state = _wait_ready("capstone-finetuned-cmp")
    print("fine-tuned deployment state:", state)
    if state == "READY":
        ft_result = _query(f"{finetuned_model_id}#accounts/{FIREWORKS_ACCOUNT_ID}/deployments/capstone-finetuned-cmp", messages, tools)
        print("fine-tuned response:", _json.dumps(ft_result.get("choices", [{}])[0].get("message", {}), indent=2))
    _delete_deployment("capstone-finetuned-cmp")

    deployments = _fw_get(f"/accounts/{FIREWORKS_ACCOUNT_ID}/deployments")
    print("\nActive deployments after cleanup:", deployments.get("totalSize", 0))


## 13. Where to go from here

- **Results across every environment/model combination actually run**, exact per-step numbers,
  and known caveats: [`RESULTS.md`](RESULTS.md).
- **Why this is built the way it is**, the two integration points, and open items (including the
  `workplace_assistant` trajectory-linkage gap): [`DESIGN.md`](DESIGN.md).
- **Adding a new NeMo Gym environment**: usually just a new `ENV_CONFIG` entry in §7 -- see
  `DESIGN.md`'s "Adding a new environment" section.
